<a href="https://colab.research.google.com/github/abduyea/Career-Trends-Analyzer/blob/main/notebooks/data_cleaning_and_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Abdulfetah Adem
# Spencer K

# Data Cleaning and Preprocessing

loads the master job-postings dataset, performs basic data cleaning, and runs exploratory data analysis (EDA) to prepare a clean datasets


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Career-Trends-Analyzer").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    mount_drive,
    load_postings,
    load_companies,
    load_jobs,
    load_mappings,
    build_master,
)
from src.config import ROOT_DIR, RAW_DIR, DATA_DIR
from src.utils import peek, missing_summary, summarize_tables

import pandas as pd

print("ROOT_DIR:", ROOT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT_DIR: /content/drive/MyDrive/Career-Trends-Analyzer


Load Data and Build Master

In [ ]:
mount_drive()

postings = load_postings()
companies = load_companies()
jobs = load_jobs()
mappings = load_mappings()

master = build_master(postings, companies, jobs, mappings)

print("Raw master shape:", master.shape)
peek(master)

# work on a copy
df = master.copy()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raw master shape: (123849, 47)
(123849, 47)


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,zip_code_company,address,url,salary_id,max_salary_salary,med_salary_salary,min_salary_salary,pay_period_salary,currency_salary,compensation_type_salary
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,07302,242 Tenth Street,https://www.linkedin.com/company/corcoran-sawy...,18531.0,20.0,NaN,17.0,HOURLY,USD,BASE_SALARY
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,NaN,NaN,8059.0,50.0,NaN,30.0,HOURLY,USD,BASE_SALARY
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,45227,6880 Wooster Pike,https://www.linkedin.com/company/the-national-...,14949.0,65000.0,NaN,45000.0,YEARLY,USD,BASE_SALARY
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,11042,3 Dakota Drive,https://www.linkedin.com/company/abrams-fenste...,11204.0,175000.0,NaN,140000.0,YEARLY,USD,BASE_SALARY
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,NaN,NaN,20809.0,80000.0,NaN,60000.0,YEARLY,USD,BASE_SALARY


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 47 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

In [ ]:
# Summary for numeric columns
df.describe(include='all').T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
job_id,123849.0,NaN,NaN,NaN,3896402138.074615,84043545.161881,921716.0,3894586595.0,3901998406.0,3904707077.0,3906267224.0
company_name,122130,24428,Liberty Healthcare and Rehabilitation Services,1108,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,123849,72521,Sales Manager,673,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,123842,107827,Position Summary: Our Sales Manager has managi...,474,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max_salary,29793.0,NaN,NaN,NaN,91939.423461,701110.138622,1.0,48.28,80000.0,140000.0,120000000.0
pay_period,36073,5,YEARLY,20628,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location,123849,8526,United States,8125,NaN,NaN,NaN,NaN,NaN,NaN,NaN
company_id,122132.0,NaN,NaN,NaN,12204012.335015,25541431.65742,1009.0,14352.0,226965.0,8047188.0,103472979.0
views,122160.0,NaN,NaN,NaN,14.618247,85.903598,1.0,3.0,4.0,8.0,9975.0
med_salary,6280.0,NaN,NaN,NaN,22015.619876,52255.873846,0.0,18.94,25.5,2510.5,750000.0


In [ ]:

def column_summary(df: pd.DataFrame) -> pd.DataFrame:
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2),
        "unique_values": df.nunique(),
        "sample_values": df.apply(lambda x: x.dropna().unique()[:5])
    })
    return summary

column_summary(df)


,dtype,missing_count,missing_percent,unique_values,sample_values
job_id,int64,0,0.00,123849,"[921716, 1829192, 10998357, 23221523, 35982263]"
company_name,object,1719,1.39,24428,"[Corcoran Sawyer Smith, The National Exemplar ..."
title,object,0,0.00,72521,"[Marketing Coordinator, Mental Health Therapis..."
description,object,7,0.01,107827,[Job descriptionA leading real estate firm in ...
max_salary,float64,94056,75.94,5321,"[20.0, 50.0, 65000.0, 175000.0, 80000.0]"
pay_period,object,87776,70.87,5,"[HOURLY, YEARLY, MONTHLY, WEEKLY, BIWEEKLY]"
location,object,0,0.00,8526,"[Princeton, NJ, Fort Collins, CO, Cincinnati, ..."
company_id,float64,1717,1.39,24474,"[2774458.0, 64896719.0, 766262.0, 1481176.0, 8..."
views,float64,1689,1.36,684,"[20.0, 1.0, 8.0, 16.0, 3.0]"
med_salary,float64,117569,94.93,1417,"[350.0, 25.0, 23.0, 56.41, 4200.0]"


In [ ]:
# Missing values count
df.isnull().sum()

# Missing value percentage
(df.isnull().mean() * 100).round(2)


,0
job_id,0.00
company_name,1.39
title,0.00
description,0.01
max_salary,75.94
pay_period,70.87
location,0.00
company_id,1.39
views,1.36
med_salary,94.93


In [ ]:
# Check true NaN / None
nan_rows = df[df.isna().any(axis=1)]
nan_rows.head()

# Check empty strings
empty_strings = (df == "").sum()
empty_strings


,0
job_id,0
company_name,0
title,0
description,0
max_salary,0
pay_period,0
location,0
company_id,0
views,0
med_salary,0


In [ ]:
print("Duplicate full rows:", df.duplicated().sum())
display(df[df.duplicated()])

if "job_id" in df.columns:
    print("Duplicate job_id:", df.duplicated("job_id").sum())
    display(df[df.duplicated("job_id")])
else:
    print("Column 'job_id' not found in this dataset.")


Duplicate full rows: 0


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,zip_code_company,address,url,salary_id,max_salary_salary,med_salary_salary,min_salary_salary,pay_period_salary,currency_salary,compensation_type_salary


Duplicate job_id: 0


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,zip_code_company,address,url,salary_id,max_salary_salary,med_salary_salary,min_salary_salary,pay_period_salary,currency_salary,compensation_type_salary


In [ ]:
df.dtypes


,0
job_id,int64
company_name,object
title,object
description,object
max_salary,float64
pay_period,object
location,object
company_id,float64
views,float64
med_salary,float64


In [ ]:
# Top 20 unique values for each column
for col in df.columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().head(20))



--- job_id ---
job_id
3906267224    1
921716        1
1829192       1
10998357      1
23221523      1
35982263      1
91700727      1
103254301     1
112576855     1
3906265303    1
3906265301    1
3906265278    1
3906265266    1
3906265264    1
3906265259    1
3906265248    1
3906265244    1
3906265243    1
3906265233    1
3906265232    1
Name: count, dtype: int64

--- company_name ---
company_name
Liberty Healthcare and Rehabilitation Services    1108
The Job Network                                   1003
J. Galt                                            604
TEKsystems                                         529
Lowe's Companies, Inc.                             527
Ingersoll Rand                                     517
Capital One                                        496
Cogent Communications                              476
Insight Global                                     418
Dice                                               415
Wells Fargo                                   

In [ ]:
# Quick outlier check using IQR
numeric_cols = df.select_dtypes(include="number").columns

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    outliers = df[(df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)]
    print(f"{col} → Outliers: {len(outliers)}")


job_id → Outliers: 718
max_salary → Outliers: 364
company_id → Outliers: 19355
views → Outliers: 16902
med_salary → Outliers: 1497
min_salary → Outliers: 293
applies → Outliers: 2897
original_listed_time → Outliers: 1924
remote_allowed → Outliers: 0
expiry → Outliers: 4384
closed_time → Outliers: 0
listed_time → Outliers: 1
sponsored → Outliers: 0
normalized_salary → Outliers: 952
zip_code → Outliers: 0
fips → Outliers: 0
company_size → Outliers: 0
salary_id → Outliers: 0
max_salary_salary → Outliers: 364
med_salary_salary → Outliers: 1497
min_salary_salary → Outliers: 293


In [ ]:
df.corr(numeric_only=True)


,job_id,max_salary,company_id,views,med_salary,min_salary,applies,original_listed_time,remote_allowed,expiry,...,listed_time,sponsored,normalized_salary,zip_code,fips,company_size,salary_id,max_salary_salary,med_salary_salary,min_salary_salary
job_id,1.000000,0.001328,-0.018856,-0.011653,0.009978,0.001595,0.003237,0.067553,NaN,-0.019619,...,0.082576,NaN,0.000953,-0.004172,0.003000,0.042581,0.064110,0.001328,0.009978,0.001595
max_salary,0.001328,1.000000,0.004865,0.002937,NaN,0.998226,-0.005922,-0.004550,NaN,0.023558,...,0.000871,NaN,0.107118,-0.004322,0.001960,-0.003585,-0.001289,1.000000,NaN,0.998226
company_id,-0.018856,0.004865,1.000000,0.029099,0.012468,0.004827,0.051471,0.002872,NaN,0.121358,...,-0.026970,NaN,-0.005125,-0.012202,-0.001075,-0.284710,-0.022115,0.004865,0.012468,0.004827
views,-0.011653,0.002937,0.029099,1.000000,0.037125,0.003579,0.494305,-0.024381,NaN,0.050233,...,-0.010586,NaN,-0.001513,-0.000920,-0.005265,-0.059783,-0.053919,0.002937,0.037125,0.003579
med_salary,0.009978,NaN,0.012468,0.037125,1.000000,NaN,-0.023131,0.004481,NaN,0.045704,...,0.001802,NaN,0.669374,-0.044134,-0.003487,-0.189337,-0.006385,NaN,1.000000,NaN
min_salary,0.001595,0.998226,0.004827,0.003579,NaN,1.000000,-0.005633,-0.002233,NaN,0.024406,...,0.001428,NaN,0.107884,-0.005512,0.001383,-0.011294,-0.001341,0.998226,NaN,1.000000
applies,0.003237,-0.005922,0.051471,0.494305,-0.023131,-0.005633,1.000000,0.018458,NaN,0.047732,...,0.016662,NaN,-0.002239,-0.002178,0.003476,-0.086322,-0.017936,-0.005922,-0.023131,-0.005633
original_listed_time,0.067553,-0.004550,0.002872,-0.024381,0.004481,-0.002233,0.018458,1.000000,NaN,0.139982,...,0.820159,NaN,-0.000132,-0.009944,-0.005577,-0.039010,0.737894,-0.004550,0.004481,-0.002233
remote_allowed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
expiry,-0.019619,0.023558,0.121358,0.050233,0.045704,0.024406,0.047732,0.139982,NaN,1.000000,...,0.164776,NaN,0.003046,0.014499,-0.019371,-0.210955,0.110991,0.023558,0.045704,0.024406


Helper functions

In [ ]:
import pandas as pd

def clean_text(x):
    """Strip whitespace; keep original case for readability."""
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    return s if s else pd.NA


def to_datetime_ms(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Convert millisecond timestamps to pandas datetime."""
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], unit="ms", errors="coerce")
    return df


Clean the master table → df_clean

In [ ]:
df_clean = df.copy()

# 1) Clean text columns: strip whitespace
text_cols = [
    "company_name",
    "title",
    "location",
    "formatted_work_type",
    "application_type",
    "posting_domain",
    "work_type",
    "formatted_experience_level",
    "state",
    "country",
    "city",
    "zip_code_company",
    "address",
]
for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(clean_text)

# 2) Fix obvious bad codes
if "country" in df_clean.columns:
    df_clean["country"] = df_clean["country"].replace("0", pd.NA)

if "zip_code_company" in df_clean.columns:
    df_clean["zip_code_company"] = df_clean["zip_code_company"].replace("0", pd.NA)

if "address" in df_clean.columns:
    df_clean["address"] = df_clean["address"].replace("0", pd.NA)

# 3) Convert millisecond timestamps → datetime
time_cols = ["original_listed_time", "listed_time", "expiry", "closed_time"]
df_clean = to_datetime_ms(df_clean, time_cols)

# 4) Drop duplicate jobs (safety)
if "job_id" in df_clean.columns:
    before = df_clean.shape[0]
    df_clean = df_clean.drop_duplicates(subset=["job_id"]).reset_index(drop=True)
    print("Dropped duplicate job_id rows:", before - df_clean.shape[0])

print("Cleaned master shape:", df_clean.shape)
df_clean.head()


Dropped duplicate job_id rows: 0
Cleaned master shape: (123849, 47)


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,zip_code_company,address,url,salary_id,max_salary_salary,med_salary_salary,min_salary_salary,pay_period_salary,currency_salary,compensation_type_salary
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,07302,242 Tenth Street,https://www.linkedin.com/company/corcoran-sawy...,18531.0,20.0,NaN,17.0,HOURLY,USD,BASE_SALARY
1,1829192,<NA>,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,<NA>,<NA>,NaN,8059.0,50.0,NaN,30.0,HOURLY,USD,BASE_SALARY
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,45227,6880 Wooster Pike,https://www.linkedin.com/company/the-national-...,14949.0,65000.0,NaN,45000.0,YEARLY,USD,BASE_SALARY
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,11042,3 Dakota Drive,https://www.linkedin.com/company/abrams-fenste...,11204.0,175000.0,NaN,140000.0,YEARLY,USD,BASE_SALARY
4,35982263,<NA>,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,<NA>,<NA>,NaN,20809.0,80000.0,NaN,60000.0,YEARLY,USD,BASE_SALARY


Save df_clean as master_cleaned

In [ ]:
from pathlib import Path
from src.config import DATA_DIR

CLEAN_DIR = DATA_DIR / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

out_path = CLEAN_DIR / "master_cleaned.csv"
df_clean.to_csv(out_path, index=False)

print("Saved cleaned master table to:", out_path)


Saved cleaned master table to: /content/drive/MyDrive/Career-Trends-Analyzer/data/clean/master_cleaned.csv
